# Online Retail — 5a, Câu 9: Tập phổ biến bằng Apriori

**Võ Thiện Linh — MSSV: 6451071041**

Tìm tập phổ biến trong các hóa đơn Germany, bỏ hóa đơn tín dụng và `POSTAGE`, dùng `min_sup = 5%`. Bản cài đặt Apriori không cần thêm thư viện ngoài Python/pandas.

In [ ]:
print('Võ Thiện Linh - MSSV: 6451071041')
from pathlib import Path
from collections import Counter
from itertools import combinations
import math
import pandas as pd
from IPython.display import display
candidates = [Path('Online Retail.xlsx'), Path('../6451071041_LAB1_Bai2_Trang37') / 'Online Retail.xlsx', Path('../Online Retail.xlsx'), Path('LAB1') / 'Phan02' / '6451071041_LAB1_Bai2_Trang37' / 'Online Retail.xlsx', Path('/content/Online Retail.xlsx')]
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError(f'Không tìm thấy Online Retail.xlsx. Thư mục đang chạy: {Path.cwd()}')
df = pd.read_excel(data_path, engine='openpyxl')
df['Description'] = df['Description'].str.strip()
df = df.dropna(subset=['InvoiceNo']).copy()
df['InvoiceNo'] = df['InvoiceNo'].astype('string').str.strip()
df = df.loc[~df['InvoiceNo'].str.contains('C', regex=False, na=False)]
germany = df.loc[df['Country'].eq('Germany')]
grouped = germany.groupby(['InvoiceNo', 'Description'], as_index=False, dropna=True)['Quantity'].sum()
invoice_ids = pd.Index(germany['InvoiceNo'].drop_duplicates(), name='InvoiceNo')
purchases = grouped.loc[grouped['Quantity'].gt(0) & grouped['Description'].ne('POSTAGE')]
items_by_invoice = purchases.groupby('InvoiceNo')['Description'].agg(frozenset).to_dict()
transactions = [items_by_invoice.get(invoice, frozenset()) for invoice in invoice_ids]
def apriori_counts(transactions, min_support):
    minimum_count = math.ceil(min_support * len(transactions))
    counts = Counter(item for transaction in transactions for item in transaction)
    previous = {frozenset([item]): count for item, count in counts.items() if count >= minimum_count}
    all_frequent = dict(previous)
    frequent_items = set(counts)
    size = 2
    while previous:
        previous_list = sorted(previous, key=lambda s: tuple(sorted(s)))
        previous_set = set(previous)
        candidates = set()
        for i, left in enumerate(previous_list):
            for right in previous_list[i + 1:]:
                candidate = left | right
                if len(candidate) == size and all(frozenset(part) in previous_set for part in combinations(candidate, size - 1)):
                    candidates.add(candidate)
        if not candidates:
            break
        candidate_counts = Counter()
        for transaction in transactions:
            for parts in combinations(sorted(transaction & frequent_items), size):
                itemset = frozenset(parts)
                if itemset in candidates:
                    candidate_counts[itemset] += 1
        previous = {itemset: count for itemset, count in candidate_counts.items() if count >= minimum_count}
        all_frequent.update(previous)
        size += 1
    return all_frequent, minimum_count
min_support = 0.05
frequent_counts, minimum_count = apriori_counts(transactions, min_support)
frequent_itemsets = pd.DataFrame([{'itemsets': ', '.join(sorted(items)), 'size': len(items), 'support': count / len(transactions), 'invoice_count': count} for items, count in frequent_counts.items()])
frequent_itemsets = frequent_itemsets.sort_values(['size', 'support'], ascending=[True, False]).reset_index(drop=True)
print(f'Hóa đơn Germany: {len(transactions):,}; min_sup = {min_support:.0%} ({minimum_count:,} hóa đơn)')
print(f'Số tập phổ biến: {len(frequent_itemsets):,}')
display(frequent_itemsets)